# KJV RAG: retrieval walkthrough

This notebook builds the **retrieval** half of a RAG (retrieval-augmented generation) system over the King James Bible, one step at a time.

The goal: given a question like *"Jonah is swallowed by a big fish"*, find the Bible passages that answer it, so an LLM can later write an answer from them and cite the verses.

| Step | What happens | Code |
|---|---|---|
| 1. Load | Read the text out of the PDF | `app/chunking.py` |
| 2. Clean and parse | Turn messy page text into 31,102 clean verses | `app/chunking.py` |
| 3. Chunk | Group verses into passages (parents) and small search windows (children) | `app/chunking.py` |
| 4. Embed | Turn every child into a vector with two different models | `app/embeddings.py` |
| 5. Store | Put the vectors in FAISS and in Chroma | `app/dense_retriever.py` |
| 6. Evaluate | Ask 15 test questions and score which model finds the right passage | `app/evaluation.py` |

The notebook only *calls* the code in `app/`, so the same steps also run from the command line:

```
python -m app.main ingest      # steps 1-3
python -m app.main embed       # steps 4-5
python -m app.main search "Jonah is swallowed by a big fish"
python -m app.main evaluate    # step 6
```

In [1]:
#quieter output: hide model-loading progress bars and Hugging Face notices
import os
import warnings

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="IProgress not found")

import transformers

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

from app import chunking, embeddings, evaluation
from app.dense_retriever import ChromaRetriever, FaissRetriever, parents_for

PDF_PATH = "data/KJVfile.pdf"
DATA_DIR = "data"


## 1. Load the PDF

The PDF has 1,517 pages. Page 1 is a table of contents; the Bible text starts on page 2.

Here is what the raw extracted text of page 2 looks like. It is readable, but not ready to use: verse numbers sit on their own lines, headings and cross-references are mixed into the text, and lines break in the middle of sentences.

In [2]:
import pymupdf

with pymupdf.open(PDF_PATH) as document:
    print(f"{len(document)} pages\n")
    print(document[1].get_text()[:600])

1517 pages

Genesis 1 
KJV  [Online] 
The Creation 
(John 1:1–5; Hebrews 11:1–3) 
1
 In the beginning God created the heaven and the earth. 2
 And the earth was without form, and void; 
and darkness [was] upon the face of the deep. And the Spirit of God moved upon the face of the 
waters.  
The First Day 
3
 And God said, Let there be light: and there was light. 4
 And God saw the light, that [it was] good: and 
God divided the light from the darkness. 5
 And God called the light Day, and the darkness he called 
Night. And the evening and the morning were the first day.  
The Second Day 
6
 And God said, 


## 2. Clean and parse into verses

Instead of generic text cleaning, we use the fact that this PDF has a **regular layout**:

| Pattern in the text | What it is | What we do |
|---|---|---|
| `Genesis 1` then `KJV  [Online]` | Chapter header | Start a new chapter |
| `1`, a newline, then a narrow no-break space | Verse number | Start a new verse |
| `The Creation`, `The First Day` | Section heading | Keep it as **metadata**, remove it from the verse text |
| `(John 1:1–5; Hebrews 11:1–3)` | Cross-reference | Remove it |
| A verse that continues on the next page | Page break | Join all pages **before** parsing |

The result is one clean record per verse: book, chapter, verse number, section heading and text.

In [3]:
full_text = chunking.read_pdf_text(PDF_PATH)
verses = chunking.parse_verses(full_text)

print(f"{len(verses)} verses")
verses[:3]

31102 verses


[{'book': 'Genesis',
  'chapter': 1,
  'verse': 1,
  'heading': 'The Creation',
  'text': 'In the beginning God created the heaven and the earth.'},
 {'book': 'Genesis',
  'chapter': 1,
  'verse': 2,
  'heading': 'The Creation',
  'text': 'And the earth was without form, and void; and darkness [was] upon the face of the deep. And the Spirit of God moved upon the face of the waters.'},
 {'book': 'Genesis',
  'chapter': 1,
  'verse': 3,
  'heading': 'The First Day',
  'text': 'And God said, Let there be light: and there was light.'}]

### Check the result

The KJV has exactly **66 books, 1,189 chapters and 31,102 verses**, so we can check the parsing against known numbers.

We also list verses that don't end in punctuation, because a heading accidentally glued onto a verse would show up there. The few that appear are real KJV verses that end without punctuation.

In [4]:
assert len({v["book"] for v in verses}) == 66
assert len({(v["book"], v["chapter"]) for v in verses}) == 1189
assert len(verses) == 31102
print("66 books, 1,189 chapters, 31,102 verses: all correct\n")

no_punctuation = [v for v in verses if v["text"][-1] not in ".,;:?!)]'" and not v["text"].endswith(chr(0x2019))]
for v in no_punctuation:
    print(f'{v["book"]} {v["chapter"]}:{v["verse"]}  ...{v["text"][-50:]}')

66 books, 1,189 chapters, 31,102 verses: all correct

Genesis 23:17  ...re] in all the borders round about, were made sure
1 Chronicles 21:11  ...nd said unto him, Thus saith the LORD, Choose thee
2 Chronicles 30:18  ...d for them, saying, The good LORD pardon every one
Psalms 96:12  ...rein: then shall all the trees of the wood rejoice
Psalms 98:8  ...ap [their] hands: let the hills be joyful together
Romans 11:7  ...ection hath obtained it, and the rest were blinded
Colossians 1:21  ...] mind by wicked works, yet now hath he reconciled


## 3. Chunk: parent-child

A single verse is often too short to carry meaning on its own, and a whole chapter is too long to search precisely. **Parent-child chunking** uses two sizes at once:

| Level | Chunking strategy | Size | Used for |
|---|---|---|---|
| **Parent** | Document-structure-based: split at book, chapter and section-heading boundaries | Up to 10 verses | What the LLM **reads** |
| **Child** | Fixed-size with overlap, inside each parent | 3 verses, overlapping by 1 | What we **search** |

At question time we search the small children (precise), then hand the LLM their parents (full context).

The overlap is only on the children: each child belongs to exactly one parent, so the LLM never gets the same verses twice.

In [5]:
parents, parent_verses = chunking.build_parents(verses)
children = chunking.build_children(parents, parent_verses)
parents_by_id = {p["id"]: p for p in parents}

print(f"{len(parents)} parents, {len(children)} children\n")

#one parent and its children, to see the overlap (each child shares one verse with the next)
parent = parents_by_id["Genesis-1-14-19"]
print("PARENT:", parent["reference"], "-", parent["heading"])
for child in children:
    if child["parent_id"] == parent["id"]:
        print("  child:", child["reference"])

4723 parents, 14618 children

PARENT: Genesis 1:14-19 - The Fourth Day
  child: Genesis 1:14-16
  child: Genesis 1:16-18
  child: Genesis 1:18-19


In [6]:
#save every level, so later steps don't need to re-read the PDF
chunking.write_jsonl(f"{DATA_DIR}/verses.jsonl", verses)
chunking.write_jsonl(f"{DATA_DIR}/parents.jsonl", parents)
chunking.write_jsonl(f"{DATA_DIR}/children.jsonl", children)

## 4. Embed

### What is an embedding?

An **embedding model** turns a piece of text into a list of numbers (a *vector*). Texts with similar meaning get vectors that point in a similar direction, so *"Jonah is swallowed by a big fish"* lands close to *"Now the LORD had prepared a great fish to swallow up Jonah"*, even though they share few words.

We embed every **child**. At question time we embed the question the same way and look for the closest children.

### The two models we compare

| | `all-MiniLM-L6-v2` | `all-mpnet-base-v2` |
|---|---|---|
| Short name here | `minilm` | `mpnet` |
| Vector size | 384 numbers | 768 numbers |
| Model size | 6 layers, ~22 million parameters | 12 layers, ~110 million parameters |
| Time to embed all 14,618 children on this CPU | **about 6 minutes** | **about 45 minutes** |
| Longest text it reads | 256 tokens (1 child gets cut off) | 384 tokens (none cut off) |

Embedding is a **one-time cost**: the vectors are saved to `data/embeddings/<model>.npy`, and `load_or_embed` reuses them on the next run. Embedding a single question takes milliseconds with either model.

Every vector is **normalized** (scaled to length 1), so "closest" simply means the highest *cosine similarity*.

In [7]:
child_texts = [c["text"] for c in children]

vectors = {}
for name in embeddings.MODELS:
    vectors[name] = embeddings.load_or_embed(name, child_texts, DATA_DIR)  # row i is the vector for children[i]
    print(f"{name}: {vectors[name].shape[0]} vectors of {vectors[name].shape[1]} numbers")

minilm: 14618 vectors of 384 numbers
mpnet: 14618 vectors of 768 numbers


## 5. Store the vectors: FAISS and Chroma

Both stores get **the same vectors**, so any difference in results would come from the store, not the model.

| | FAISS | Chroma |
|---|---|---|
| What it is | A fast search index, nothing more | A database: vectors plus each child's text and metadata |
| How it searches | Exact: compares the question with every child | Approximate (HNSW): almost always the same answer, much faster on huge collections |
| Finding the parent | We look up the child ourselves | `parent_id` is stored next to each vector |
| Saved to | `data/faiss/<model>.index` | `data/chroma/` |

A search does three things: embed the question with the same model, find the closest children, then return their **parents** with `parents_for()`.

In [8]:
import chromadb

chroma = chromadb.PersistentClient(path=f"{DATA_DIR}/chroma")

retrievers = {}
for name in embeddings.MODELS:
    encode = embeddings.encoder(name)
    retrievers[("FAISS", name)] = FaissRetriever.build(vectors[name], children, encode)
    retrievers[("Chroma", name)] = ChromaRetriever.build(chroma, f"kjv_children_{name}", vectors[name], children, encode)


def search(store, model, question, k=3):
    return parents_for(retrievers[(store, model)].search(question), parents_by_id, k=k)


for passage in search("FAISS", "mpnet", "Jonah is swallowed by a big fish"):
    print(passage["text"][:200], "...\n")

Jonah 1:11-17 (Jonah Cast into the Sea)
11 Then said they unto him, What shall we do unto thee, that the sea may be calm unto us? for the sea wrought, and was tempestuous. 12 And he said unto them, Ta ...

Jonah 2:1-10 (Jonah’s Prayer)
1 Then Jonah prayed unto the LORD his God out of the fish’s belly, 2 And said, I cried by reason of mine affliction unto the LORD, and he heard me; out of the belly of he ...

Jonah 1:4-10 (The Great Storm)
4 But the LORD sent out a great wind into the sea, and there was a mighty tempest in the sea, so that the ship was like to be broken. 5 Then the mariners were afraid, an ...



## 6. Which model works best?

To compare fairly, we ask every model and store the same **15 test questions** from `data/test_questions.json`. For each one we already know the verse that answers it. Some questions are close to the KJV wording; others are everyday paraphrases, because real users rarely type "thou" and "hath".

Each search returns a ranked list of passages. We check **where the right passage appears** in that list:

| Score | What it means | Best possible |
|---|---|---|
| **correct at #1** | Questions where the right passage was the very first result | 15 |
| **correct in top 5** | Questions where the right passage was somewhere in the first 5 results | 15 |
| **MRR** (mean reciprocal rank) | One number that rewards finding it *early*: 1st place scores 1, 2nd scores 1/2, 3rd scores 1/3, a miss scores 0, then we average | 1.0 |

In [9]:
import pandas as pd

questions = evaluation.load_test_questions(f"{DATA_DIR}/test_questions.json")

rows, ranks = [], {}
for (store, name), retriever in retrievers.items():
    scores, ranks[(store, name)] = evaluation.evaluate(lambda q: parents_for(retriever.search(q), parents_by_id), questions)
    rows.append({"store": store, "model": name, **scores})

pd.DataFrame(rows)

,store,model,correct at #1,correct in top 5,MRR
0,FAISS,minilm,7,14,0.652
1,Chroma,minilm,7,14,0.652
2,FAISS,mpnet,9,14,0.719
3,Chroma,mpnet,9,14,0.719


### Question by question

The position (1 = first result) at which each model found the right passage, using FAISS:

In [10]:
pd.DataFrame({
    "question": [q["question"] for q in questions],
    "expected verse": [f'{q["book"]} {q["chapter"]}:{q["verse"]}' for q in questions],
    **{name: [r if r else "missed" for r in ranks[("FAISS", name)]] for name in embeddings.MODELS},
})

,question,expected verse,minilm,mpnet
0,Am I my brother's keeper?,Genesis 4:9,3,4
1,For God so loved the world,John 3:16,missed,missed
2,The Lord is my shepherd,Psalms 23:1,1,1
3,Thou shalt not kill,Exodus 20:13,2,3
4,Blessed are the poor in spirit,Matthew 5:3,1,1
5,Jesus wept,John 11:35,4,1
6,What is love? Love is patient and kind,1 Corinthians 13:4,5,5
7,Jesus feeds five thousand people with loaves a...,Matthew 14:21,2,1
8,The story of the prodigal son who left home,Luke 15:11,1,1
9,Jonah is swallowed by a big fish,Jonah 1:17,1,1


## Results

### The winner: `all-mpnet-base-v2`

| Model | Correct at #1 | Correct in top 5 | MRR | Time to embed |
|---|---|---|---|---|
| `minilm` | 7 of 15 | 14 of 15 | 0.652 | ~6 min |
| **`mpnet`** | **9 of 15** | **14 of 15** | **0.719** | ~45 min |

- **mpnet puts the right passage first more often** (9 vs 7). It clearly wins on *"Jesus wept"* (1st vs 4th) and *"Jesus feeds five thousand"* (1st vs 2nd).
- **Both models find almost everything in the top 5** (14 of 15), so for an LLM that reads the top 5 passages the gap is smaller than the #1 score suggests.
- **FAISS and Chroma give identical scores** for each model. That confirms both stores hold the same vectors and are set up correctly, and that Chroma's approximate search loses nothing at this size.
- **mpnet costs more**: about 7x longer to embed on this CPU and twice the storage. Because embedding is a one-time cost and a question still takes milliseconds, that cost is worth paying.

**Recommendation: use `all-mpnet-base-v2`**, stored in Chroma (it keeps the text and `parent_id` next to each vector).

### What both models get wrong, and why

| Question | What happened | Why |
|---|---|---|
| *"For God so loved the world"* | **Both missed John 3:16** | Dense models match *meaning*, not exact words. They return other passages about God, love and the world (e.g. 1 John 2:15, *"Love not the world"*). A famous quote is exactly what **keyword search** is good at. |
| *"What is love? Love is patient and kind"* | Found 1 Corinthians 13 only at #5 | The KJV says **"charity"**, not "love": *"Charity suffereth long, and is kind"*. Both models prefer 1 John 4, *"God is love"*, which is a reasonable answer too. |
| *"Thou shalt not kill"* | Exodus 20 at #2 or #3 | The Ten Commandments appear **twice** (Exodus 20 and Deuteronomy 5). The other results are correct passages, just not the one our test expects. |

### What's next

These misses point to the next steps in the project structure:
1. **Sparse retrieval (BM25)**, `app/sparse_retriever.py`: keyword search that should catch exact quotes like *"For God so loved the world"*.
2. **Hybrid retrieval**, `app/hybrid_retriever.py`: combine dense (meaning) and sparse (exact words) results.
3. **MMR** and a **reranker**, `app/mmr.py` and `app/reranker.py`: remove near-duplicate passages, then re-score the best candidates more carefully.